In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("./dataset/LeesmateriaalCSV.csv", sep=";")

In [157]:
# Controleren of Themas_boek en trefwoorden per rij identiek zijn
df["Themas_boek"] = (
    df["Themas_boek"]
    .astype("string")
    .fillna("")
    .str.replace(r"(?:https?://|www\.)\S+(?:\s+\[[^\]]+\])?", "", regex=True, case=False)
    .str.replace(r"\[[^\]]+\]", "", regex=True)
    .str.strip(" ,;")
)
themas_boek = df["Themas_boek"].astype("string").fillna("").map(
    lambda waarde: ";".join(woord.strip() for woord in waarde.split(";"))
)
trefwoorden = df["trefwoorden"].astype("string").fillna("").map(
    lambda waarde: ";".join(woord.strip() for woord in waarde.split(";"))
)
zelfde = themas_boek.eq(trefwoorden)
verschillen = df.loc[~zelfde, ["Titel", "Themas_boek", "trefwoorden"]]

print(f"Alle waarden identiek: {verschillen.empty}")
verschillen

Alle waarden identiek: False


,Titel,Themas_boek,trefwoorden
154,De wondere wereld van nachtvlinders,,[naturetoday.com]
155,Boeren geven de zwarte stern vleugels in de Ut...,,[naturetoday.com]
188,Zolang het tuin duurt 2,"zelfreflectie,doorzetten",zelfreflectie; doorzetten
201,De kleine kust,"jongeren,vriendschap",jongeren; vriendschap
231,Wat Joris niet wist,"doorzetten,liefde",doorzetten; liefde
242,Sleutel zonder kust,liefde/samenleving,liefde; samenleving
246,De brief van Jelle,geheimen|macht,geheimen; macht
254,Een foto voor Sem,"veerkracht,oorlog",veerkracht; oorlog
270,Het witte huis,reizen/oorlog,reizen; oorlog
282,Grens in de haven,"doorzetten,stalking",doorzetten; stalking


In [158]:
# Controleren of elk onderwerp in de trefwoorden voorkomt
onderwerpen = df["onderwerp"].astype("string").fillna("").str.strip().str.casefold()
trefwoorden = [
    [woord.strip().casefold() for woord in rij.split(";") if woord.strip()]
    for rij in df["trefwoorden"].astype("string").fillna("")
]
onderwerp_gevonden = pd.Series(
    [
        bool(onderwerp) and onderwerp in woorden
        for onderwerp, woorden in zip(onderwerpen, trefwoorden)
    ],
    index=df.index,
)
ontbreekt = df.loc[
    onderwerpen.ne("") & ~onderwerp_gevonden,
    ["Titel", "onderwerp", "trefwoorden"],
]
print(f"Alle onderwerpen staan in trefwoorden: {ontbreekt.empty}")
ontbreekt

Alle onderwerpen staan in trefwoorden: True


,Titel,onderwerp,trefwoorden


In [159]:
# Controleren of ingevulde genres_kopie al in genre staan
genre = df["genre"].astype("string").fillna("").str.strip().str.casefold()
genre_kopie = df["genre_kopie"].astype("string").fillna("").str.strip().str.casefold()
extra_genres = genre_kopie.ne("") & genre_kopie.ne(genre)
verschillen_genre = df.loc[extra_genres, ["Titel", "genre", "genre_kopie"]]

print(f"Alle ingevulde genres_kopie komen overeen met genre: {verschillen_genre.empty}")
verschillen_genre

Alle ingevulde genres_kopie komen overeen met genre: True


,Titel,genre,genre_kopie


In [5]:
#onnodige kolommen verwijderen
df = df.drop(columns=['Unnamed: 3', 'Themas_boek', 'onderwerp', 'materiaal_id', 'interne_batchcode', 'genre_kopie', 'woordenaantal_omschrijving', 'aandeel_lange_woorden_pct', 'aantal_zinnen', 'administratieve_prioriteit'])

In [6]:
# Genre uit Type materiaal verplaatsen als genre leeg is
mask = df["Type materiaal"].str.match(r"(?i)^boek\s*-", na=False)
genre = df.loc[mask, "Type materiaal"].str.split("-", n=1).str[1].str.strip()
missing_genre = df["genre"].isna()
df.loc[mask & missing_genre, "genre"] = genre.loc[mask & missing_genre]

In [7]:
#Type materiaal normaliseren
mask = df["Type materiaal"].str.match(r"(?i)^boek\s*-", na=False)
df.loc[mask, "Type materiaal"] = "boek"
df.loc[df["Type materiaal"] == "Boek", "Type materiaal"] = "boek"
df.loc[df["Type materiaal"] == "digitale krant", "Type materiaal"] = "krant (digitaal)"

In [8]:
#Taal normaliseren
df["taal"] = "NL"

In [9]:
#Doelgroep normaliseren
df.loc[df["doelgroep"] == "mbo", "doelgroep"] = "MBO"
df.loc[df["doelgroep"] == "mbo / nt2", "doelgroep"] = "MBO/NT2"

In [10]:
#Doelgroepen combineren
def combineer_doelgroepen(row):
    groepen = []

    for waarde in row.dropna():
        for groep in str(waarde).split("/"):
            groep = groep.strip()

            if groep != "" and groep not in groepen:
                groepen.append(groep)

    volgorde = {"MBO": 0, "NT2": 1}
    groepen.sort(key=lambda groep: volgorde.get(groep, 2))

    return "/".join(groepen)

df["doelgroep"] = df[["doelgroep", "doelgroep_extra"]].apply(
    combineer_doelgroepen, axis=1
)
df.loc[df["doelgroep"] == "", "doelgroep"] = np.nan

df = df.drop(["doelgroep_extra"], axis=1)

In [11]:
#onrealistische waarden in gem_woordlengte vervangen door NaN
df.loc[df["gem_woordlengte"] > 10, ["gem_woordlengte"]] = np.nan

In [12]:
#interessecluster normaliseren
df["interessecluster"] = df["interessecluster"].str.lower()

In [13]:
#onrealistische en onbekende jaartallen in publicatiejaar vervangen door 'onbekend'
df.loc[df["publicatiejaar"] == 'onbekend', ["publicatiejaar"]] = np.nan
df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")
df.loc[df["publicatiejaar"] > 2026, ["publicatiejaar"]] = np.nan

In [14]:
#beschikbaar_formaat normaliseren
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.lower()

In [15]:
#aantal_paginas normaliseren
df.loc[df["aantal_paginas"] < 1, "aantal_paginas"] = np.nan
df["aantal_paginas"] = df["aantal_paginas"].astype('Int64')

In [16]:
#geschatte_leestijd_min normaliseren
df.loc[df["geschatte_leestijd_min"] < 1, "geschatte_leestijd_min"] = np.nan

In [17]:
#Gegevensstatus opschonen en veranderen naar bruikbare data
df.loc[df["gegevensstatus"] == "Aangevuld (deels gesimuleerd)", ["gegevensstatus"]] = 'Beschikbaar'
df.loc[df["gegevensstatus"] == "gecontroleerd", ["gegevensstatus"]] = 'Binnenkort Terug Verwacht'
df.loc[df["gegevensstatus"] == "In behandeling", ["gegevensstatus"]] = 'Uitgeleend'
df.loc[df["gegevensstatus"].isna(), ["gegevensstatus"]] = 'Onbekend'

In [18]:
# Korte url-vermelding uit bron_url verwijderen
df["bron_url"] = df["bron_url"].str.replace(
    r"((?:https?://|www\.)\S+?)(?:\s*,?\s*\[[^\]]+\])+",
    r"\1",
    regex=True,
)

In [19]:
# Auteurs scheiden met een slash
df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s+en\s+", "/", case=False, regex=True)
    .str.replace(r"\s*[,&]\s*", "/", regex=True)
    .str.replace(r"\s*/\s*", "/", regex=True)
    .str.strip()
)

In [20]:
df["trefwoorden"] = df["trefwoorden"].str.replace("; ", ";")

In [ ]:
df

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,...,gegevensstatus,gem_woordlengte,gem_zinslengte_woorden,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,Gemiddelde beoordeling
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII;onderduik,NL,265,556.0,...,Beschikbaar,NaN,NaN,NaN,verhalen,Educatief Media,2025,pdf,CAT-00001,3.4
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,boek,3F,fictie,spanning;stalking,NL,171,184.0,...,Beschikbaar,5.54,14.0,48.8,verhalen,Leeswereld,2005,digitaal,CAT-00002,2.5
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",boek,2F,fictie,liefde;relaties,NL,219,662.0,...,Beschikbaar,6.35,11.5,49.2,verhalen,NaN,2025,digitaal,CAT-00003,3.8
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,boek,3F,fictie,ontmoeting;verbondenheid,NL,288,748.0,...,Beschikbaar,7.35,11.5,55.3,verhalen,Uitgeverij Noord,2018,digitaal,CAT-00004,3.9
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,boek,2F,fictie,geheimen;schuld,NL,358,196.0,...,Beschikbaar,6.48,10.5,49.2,verhalen,Uitgeverij Noord,<NA>,online,CAT-00005,3.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,Zorgen om studiefinanciering: 'aanpak nodig',Lars Dekker (NOS),"criminaliteit, klimaat, media",blogpost,3F,informatief,NaN,NL,3,14.0,...,Beschikbaar,8.33,3.0,51.8,verhalen,NaN,2013,online,CAT-00279,NaN
279,Wat betekent jeugdcriminaliteit voor jou?,Daan de Vries (NOS),"werk, AI, privacy, jongeren",online artikel,3F,informatief,NaN,NL,6,19.0,...,Beschikbaar,5.25,4.0,36.7,verhalen,Educatief Media,<NA>,online,CAT-00280,NaN
280,Zolang het belofte duurt,Vera Prins,Een man van veertig wordt geconfronteerd met d...,boek,2F,fictie,NaN,NL,145,245.0,...,Uitgeleend,5.53,9.5,32.6,verhalen,NaN,2005,pdf,CAT-00281,NaN
281,Zomer zonder grens,Jelle Meijer,"Toegankelijk verhaal over onderzoek, burgersch...",boek,2F,thriller,onderzoek;burgerschap,NL,255,346.0,...,Beschikbaar,5.47,9.5,34.3,verhalen,Leeswereld,1998,papier,CAT-00282,NaN


In [22]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 22)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,str,6
Auteur,231,7,str,45
Korte omschrijving,282,0,str,1
Type materiaal,7,0,str,276
Niveau,4,6,str,273
genre,5,15,str,263
trefwoorden,93,181,str,9
taal,1,0,str,282
aantal_paginas,157,28,Int64,98
geschatte_leestijd_min,212,9,float64,62


In [ ]:
kolom = "bron_url"

df[[kolom, "Titel"]][df[kolom].duplicated(keep=False) & df[kolom].notna()].sort_values(kolom)